# Genotype–Phenotype Heterogeneity in Non-Classical 21-Hydroxylase Deficiency - Exploration with `mlcroissant`
This notebook demonstrates how to load, overview, extract, and process the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is structured via a Croissant schema, available from the following URL:

https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print overview
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\nIdentifier: {metadata.identifier}\nVersion: {metadata.version}")

## 2. Data Overview
Inspect available record sets, fields, and their `@id`s.

In [ ]:
# List out all available record sets
record_sets = dataset.record_sets

print("Available Record Sets:")
for record_set in record_sets:
    print(f"- @id: {record_set.id}, name: {getattr(record_set, 'name', 'N/A')}")

# Show the available fields in each record set
for record_set in record_sets:
    print(f"\nFields for Record Set @id {record_set.id}:")
    for field in record_set.fields:
        print(f"  - @id: {field.id}, name: {getattr(field, 'name', 'N/A')}, dataType: {getattr(field, 'dataType', 'N/A')}")

## 3. Data Extraction
Load data from each record set into Pandas DataFrames for further analysis.

We use record set and field `@id`s for targeting data.

In [ ]:
# Extract data from each record set

record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns for record set @id '{record_set_id}':")
    print(df.columns.tolist())
    print(f"Sample data:")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply processing steps: filter, normalize, categorize, group.
We'll choose one record set with numerical fields as example.


In [ ]:
# Choose the record set and numeric field to analyze

# Find the record set with hormone levels (typically Table 2)
record_set_names = {rs.id: getattr(rs, 'name', '').lower() for rs in dataset.record_sets}
hormones_rs_id = next((rsid for rsid, name in record_set_names.items() if 'hormone' in name), list(record_set_names.keys())[0])

df = dataframes[hormones_rs_id]
print(f"Using record set @id: {hormones_rs_id} for hormone level analysis.")

# Show all columns
print("Columns available:", df.columns.tolist())

# Pick a numeric hormone field
numeric_fields = [col for col in df.columns if 'level' in col or 'concentration' in col or 'amount' in col or df[col].dtype in ['int64','float64']]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    numeric_field_id = df.columns[0]

# Set a threshold for filtering
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by another categorical field (e.g., 'adrenal_ct_result' or first non-numeric column)
cat_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
group_field_id = cat_fields[0] if cat_fields else numeric_field_id
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field_id} (mean values):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we plot the normalized hormone levels.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot normalized hormone levels by group (if available)
if group_field_id in filtered_df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field_id, y=f"{numeric_field_id}_normalized", data=filtered_df)
    plt.title(f"Normalized {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"{numeric_field_id} (normalized)")
    plt.xlabel(group_field_id)
    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(6,4))
    plt.hist(filtered_df[f"{numeric_field_id}_normalized"], bins=10, color='skyblue')
    plt.title(f"Distribution of Normalized {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id} (normalized)")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we've loaded the FAIR² dataset with `mlcroissant`, explored its metadata and table structure using `@id` references, extracted tabular data for analysis, performed basic filtering and normalization on hormone levels, and visualized values grouped by clinical imaging results or other variables.

This illustrates how Croissant datasets can be processed programmatically and highlights the importance of using consistent `@id` references for robust data exploration. For further domain-specific analyses, consult the dataset documentation or schema for additional clinical and genetic variables.

**Note:** The small and specific sample size of the dataset limits the scope for statistical inference and predictive modeling.